# GPT-2 with LoRA Finetuning

In [1]:
# required imports
import os
import sys
import numpy as np
import pandas as pd
import json
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import os
from transformers import AutoModelForCausalLM, AutoConfig
import math
from torch.optim import Adam, AdamW, lr_scheduler
import torch.nn.functional as F
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
model_name = 'gpt2'

## Prepare Dataset and Dataloader

Here we use the dataset used in the official [official LoRA example](https://github.com/microsoft/LoRA).

In [4]:
# downloading the dataset (not working on Windows)
# !wget https://github.com/microsoft/LoRA/raw/main/examples/NLG/data/e2e/train.txt -O ../data/train.txt

In [5]:
# !wget https://github.com/microsoft/LoRA/raw/main/examples/NLG/data/e2e/test.txt -O ../data/test.txt

### Some EDA

In [6]:
train_data  = pd.read_csv('../data/train.txt', sep='\t', header=None)
display(train_data.head())
display(train_data.describe())

,0
0,name : The Vaults | Type : pub | price : more ...
1,name : The Cambridge Blue | Type : pub | food ...
2,name : The Eagle | Type : coffee shop | food :...
3,name : The Mill | Type : coffee shop | food : ...
4,name : Loch Fyne | food : French | customer ra...


,0
count,42061
unique,40859
top,name : Strada | Type : restaurant | customer r...
freq,5


### Preprocess the data

In [7]:
def dump_json(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as reader, open(output_file, 'w', encoding='utf-8') as writer:
        for line in reader:
            items = line.strip().split('||')
            context = items[0]
            response = items[1].strip('\n')
            x = {'context': context, 'response': response}
            writer.write(json.dumps(x) + '\n')


In [8]:
dump_json('../data/train.txt', '../data/train.json')
dump_json('../data/test.txt', '../data/test.json')

In [9]:
# check the json files
with open('../data/train.json', 'r') as reader:
    for _ in range(5):
        print(json.loads(reader.readline()))

{'context': 'name : The Vaults | Type : pub | price : more than £ 30 | customer rating : 5 out of 5 | near : Café Adriatic', 'response': 'The Vaults pub near Café Adriatic has a 5 star rating . Prices start at £ 30 .'}
{'context': 'name : The Cambridge Blue | Type : pub | food : English | price : cheap | near : Café Brazil', 'response': 'Close to Café Brazil , The Cambridge Blue pub serves delicious Tuscan Beef for the cheap price of £ 10.50 . Delicious Pub food .'}
{'context': 'name : The Eagle | Type : coffee shop | food : Japanese | price : less than £ 20 | customer rating : low | area : riverside | family friendly : yes | near : Burger King', 'response': 'The Eagle is a low rated coffee shop near Burger King and the riverside that is family friendly and is less than £ 20 for Japanese food .'}
{'context': 'name : The Mill | Type : coffee shop | food : French | price : £ 20 - 25 | area : riverside | near : The Sorrento', 'response': 'Located near The Sorrento is a French Theme eatery

#### Tokenization

In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    fast_tokenizer=True)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # to avoid warnings

In [11]:
context_size = 512  # for GPT-2, the maximum context size is 1024

print(f'Maximum token length: {tokenizer.model_max_length}')
print(f'Training context size: {context_size}')

Maximum token length: 1024
Training context size: 512


### Dataset and Dataloader

Prepare the dataset and dataloader for training.

In [12]:
def fill_ignore_label(label, context_tokens):
    label[:len(context_tokens) - 1] = [-100] * (len(context_tokens) - 1)  # -100 is the ignore label as in PyTorch CrossEntropyLoss
    return label

In [13]:
def pad_tokens(tokens, max_seq_length, padding_token):
    padded_tokens = tokens[:max_seq_length]
    token_len = len(padded_tokens)
    padded_tokens = padded_tokens + \
        [padding_token for _ in range(max_seq_length - token_len)]
    return padded_tokens

In [14]:
def collate_batch(batch):
    # tokenize both context and response respectively
    # (context and response is delimited by "\n")
    context_list = list(zip(*batch))[0]
    context_list = [context + "\n" for context in context_list]
    response_list = list(zip(*batch))[1]
    context_tokenized = tokenizer(context_list)
    context_tokens = context_tokenized["input_ids"]
    context_masks = context_tokenized["attention_mask"]
    response_tokenized = tokenizer(response_list)
    response_tokens = response_tokenized["input_ids"]
    response_masks = response_tokenized["attention_mask"]
    # concatenate token
    inputs = [i + j for i, j in zip(context_tokens, response_tokens)]
    masks = [i + j for i, j in zip(context_masks, response_masks)]
    # create label
    eos_id = tokenizer.encode(tokenizer.eos_token)[0]  # get the id of the end-of-sentence token
    labels = [token[1:] + [eos_id] for token in inputs]
    labels = list(map(fill_ignore_label, labels, context_tokens))  # fill ignore label for context tokens, except the last token
    # truncate and pad tokens to make sure the length of tokens is context_size
    inputs = [pad_tokens(t, context_size, 0) for t in inputs]  # OPT and GPT-2 doesn't use pad token (instead attn mask is used)
    masks = [pad_tokens(t, context_size, 0) for t in masks]
    labels = [pad_tokens(t, context_size, -100) for t in labels]
    # convert to tensor
    inputs = torch.tensor(inputs, dtype=torch.int64).to(device)
    masks = torch.tensor(masks, dtype=torch.int64).to(device)
    labels = torch.tensor(labels, dtype=torch.int64).to(device)
    return inputs, labels, masks

In [15]:
class LanguageModelDataset(Dataset):
    def __init__(self, data_path):
        self.dataset = pd.read_json(data_path, lines=True)
        self.data = list(zip(self.dataset['context'], self.dataset['response']))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [16]:
train_dataset = LanguageModelDataset('../data/train.json')
test_dataset = LanguageModelDataset('../data/test.json')

In [17]:
batch_size = 8

In [18]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_batch)

In [19]:
# check the dataloader
for inputs, labels, masks in train_dataloader:
    print(inputs.size(), labels.size(), masks.size())
    break

torch.Size([8, 512]) torch.Size([8, 512]) torch.Size([8, 512])


## Model

In [20]:
config = AutoConfig.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name, config=config).to(device)

In [21]:
# check the model
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


### Downstream Text Generation

In [22]:
def generate_text(model, input_ids, attention_mask, eos_id, pred_sequence_length):
    predicted_last_id = -1
    start_token_len = torch.sum(attention_mask).cpu().numpy()
    token_len = start_token_len
    with torch.no_grad():
        while (predicted_last_id != eos_id) and \
              (token_len - start_token_len < pred_sequence_length):
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            predicted_ids = torch.argmax(output.logits, dim=-1).cpu().numpy()
            predicted_last_id = predicted_ids[0][token_len - 1]
            input_ids[0][token_len] = predicted_last_id
            attention_mask[0][token_len] = 1
            token_len = torch.sum(attention_mask).cpu().numpy()
    return input_ids, token_len

In [23]:
eos_id = tokenizer.encode(tokenizer.eos_token)[0]

tokenized1 = tokenizer("Once upon a time,")
input_ids_tokenized1 = tokenized1["input_ids"]
attention_mask_tokenized1 = tokenized1["attention_mask"]
input_ids1 = pad_tokens(input_ids_tokenized1, context_size, 0)
attention_mask1 = pad_tokens(attention_mask_tokenized1, context_size, 0)
input_ids1 = torch.tensor([input_ids1], dtype=torch.int64).to(device)
attention_mask1 = torch.tensor([attention_mask1], dtype=torch.int64).to(device)

result_token1, result_len1 = generate_text(model, input_ids1, attention_mask1, eos_id, pred_sequence_length=100)
print(tokenizer.decode(result_token1[0][:result_len1]))

Once upon a time, the world was a place of great beauty and great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world


In [24]:
eos_id = tokenizer.encode(tokenizer.eos_token)[0]

tokenized2 = tokenizer("My name is Chadman and  I am")
input_ids_tokenized2 = tokenized2["input_ids"]
attention_mask_tokenized2 = tokenized2["attention_mask"]
input_ids2 = pad_tokens(input_ids_tokenized2, context_size, 0)
attention_mask2 = pad_tokens(attention_mask_tokenized2, context_size, 0)
input_ids2 = torch.tensor([input_ids2], dtype=torch.int64).to(device)
attention_mask2 = torch.tensor([attention_mask2], dtype=torch.int64).to(device)

result_token2, result_len2 = generate_text(model, input_ids2, attention_mask2, eos_id, pred_sequence_length=100)
print(tokenizer.decode(result_token2[0][:result_len2]))

My name is Chadman and  I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler. I am a professional wrestler.


In [25]:
for idx, (input_ids, _, attention_mask) in enumerate(test_dataloader):
    print(f"Test data {idx + 1}")
    input_len = torch.sum(attention_mask).cpu().numpy()
    print(tokenizer.decode(input_ids[0][:input_len]))
    result_token, result_len = generate_text(model, input_ids, attention_mask, eos_id, pred_sequence_length=30)
    print(tokenizer.decode(result_token[0][:result_len]))
    print()
    if idx == 4:
        break

Test data 1
name : Blue Spice | Type : coffee shop | area : city centre
A coffee shop in the city centre area called Blue Spice .
name : Blue Spice | Type : coffee shop | area : city centre
A coffee shop in the city centre area called Blue Spice . It is located in the centre of the city centre.
Blue Spice is a coffee shop in the city centre area.
Blue Spice is a coffee

Test data 2
name : Blue Spice | Type : coffee shop | area : city centre
Blue Spice is a coffee shop in city centre .
name : Blue Spice | Type : coffee shop | area : city centre
Blue Spice is a coffee shop in city centre . It is located in the centre of the city centre.
Blue Spice is a coffee shop in city centre . It is located in the centre of the

Test data 3
name : Blue Spice | Type : coffee shop | area : riverside
There is a coffee shop Blue Spice in the riverside area .
name : Blue Spice | Type : coffee shop | area : riverside
There is a coffee shop Blue Spice in the riverside area . It is located in the middle of t

## LoRA

In [26]:
class LoRALinear(nn.Module):
    def __init__(self, weight, bias, rank):
        super(LoRALinear, self).__init__()
        m, n = weight.shape

        if bias is None:
            self.linear = nn.Linear(n, m, bias=False)
            self.liear.load_state_dict({'weight': weight})
        else:
            self.linear = nn.Linear(n, m)
            self.linear.load_state_dict({'weight': weight, 'bias': bias})

        self.A = nn.Parameter(torch.zeros(n, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(rank, m))

    def forward(self, inputs):
        x = self.linear(inputs)
        y =  inputs @ self.A @ self.B
        return x + y

In [27]:
rank = 64

In [28]:
# get target modules
target_modules = []
for name, module in model.named_modules():
    if "attn.c_attn" in name:
        target_modules.append(name)

# replace target modules with LoRALinear
for module_name in target_modules:
    name_segments = module_name.split('.')
    module_list = [model]
    for seg in name_segments:
        module_list.append(getattr(module_list[-1], seg))

    lora = LoRALinear(module_list[-1].weight.transpose(0, 1), module_list[-1].bias, rank).to(device)
    setattr(module_list[-2], name_segments[-1], lora)  # replace the module

In [29]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): LoRALinear(
            (linear): Linear(in_features=768, out_features=2304, bias=True)
          )
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_feature

In [30]:
# freeze the model except the LoRALinear layers
for name, param in model.named_parameters():
    if "A" in name or "B" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [31]:
# compute number of trainable parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {num_params}")

Number of trainable parameters: 2359296


## Fine-tuning

In [32]:
optimizer = AdamW(model.parameters(), betas=(0.9, 0.99), eps=1e-9)

In [33]:
num_epochs = 1
num_warmup_steps = 1000
gradient_accumulation_steps = 16
num_training_steps = math.ceil(len(train_dataloader) / batch_size / gradient_accumulation_steps)

In [34]:
def _get_linear_schedule_with_warmup(current_step):
    if current_step < num_warmup_steps:
        return float(current_step) / float(max(1, num_warmup_steps))
    return max(0.0, float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps)))

In [35]:
scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=_get_linear_schedule_with_warmup)

In [36]:
def train_step(model, optimizer, scheduler, inputs, labels, masks):
    model.train()
    optimizer.zero_grad()
    outputs = model(
        input_ids=inputs,
        attention_mask=masks
    )
    loss = F.cross_entropy(outputs.logits.transpose(1, 2), labels)
    loss.backward()
    return loss.item()

In [37]:
def eval_step(model, inputs, labels, masks):
    model.eval()
    with torch.no_grad():
        outputs = model(
            input_ids=inputs,
            attention_mask=masks
        )
        loss = F.cross_entropy(outputs.logits.transpose(1, 2), labels)
    return loss.item()

In [38]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}")
    train_loss = 0
    with tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch + 1}") as pbar:
        for idx, (inputs, labels, masks) in pbar:
            inputs = inputs.to(device)
            labels = labels.to(device)
            masks = masks.to(device)

            with torch.set_grad_enabled(True):
                loss = train_step(model, optimizer, scheduler, inputs, labels, masks)
                train_loss += loss
                pbar.set_postfix({'loss': loss})

                if ((idx + 1) % gradient_accumulation_steps) == 0 or idx == len(train_dataloader) - 1:
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                pbar.set_postfix({'loss': train_loss / (idx + 1)})
                pbar.update(1)

            with open('loss.txt', 'a') as f:
                f.write(f"{loss}\n")

    # evaluation
    test_loss = 0
    with tqdm(enumerate(test_dataloader), total=len(test_dataloader), desc=f"Test") as pbar:
        for idx, (inputs, labels, masks) in pbar:
            inputs = inputs.to(device)
            labels = labels.to(device)
            masks = masks.to(device)

            with torch.set_grad_enabled(False):
                loss = eval_step(model, inputs, labels, masks)
                test_loss += loss
                pbar.set_postfix({'loss': loss})
                pbar.update(1)

    # save the model
    torch.save(model.state_dict(), f"lora_gpt2_{epoch + 1}_{rank}.pt")

Epoch 1


Test: 100%|██████████| 4693/4693 [04:02<00:00, 19.37it/s, loss=0.626]
